In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu

In [ ]:
import pandas as pd
import numpy as np
import torch
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

In [ ]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
chunksize = 10000

data = pd.read_json(
    "/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json",
    lines=True,
    chunksize=chunksize
)

In [ ]:
allowed_categories = [
    'cs.AI',
    'cs.LG',
    'stat.ML',
    'cs.CL',
    'cs.IR',
    'cs.CV',
    'cs.NE'
]

In [ ]:
papers = []

for chunk in data:

    filtered = chunk[
        chunk['categories'].apply(
            lambda x: any(
                cat in x
                for cat in allowed_categories
            )
        )
    ]

    papers.append(filtered)

In [ ]:
df = pd.concat(
    papers,
    ignore_index=True
)

print("Jumlah awal paper:")
print(len(df))

In [ ]:
df['update_date'] = pd.to_datetime(
    df['update_date']
)

df = df[
    df['update_date'].dt.year >= 2017
]

print("Rentang tahun:")
print(df['update_date'].dt.year.min())
print(df['update_date'].dt.year.max())

print()
print("Jumlah paper modern:")
print(len(df))

In [ ]:
df = df[
    [
        'title',
        'abstract',
        'categories',
        'update_date'
    ]
]

In [ ]:
df = df.dropna()
df = df.reset_index(drop=True)

print("Jumlah paper relevan:")
print(len(df))

df.head()

In [ ]:
df['text'] = (
    "passage: "
    + df['title']
    + ". Abstract: "
    + df['abstract']
)

In [ ]:
model = SentenceTransformer(
    'BAAI/bge-base-en-v1.5'
)

In [ ]:
embeddings = model.encode(
    df['text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

In [ ]:
embeddings = np.array(
    embeddings
).astype('float32')

print(embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total vector dalam FAISS:")
print(index.ntotal)

In [ ]:
np.save(
    "embeddings.npy",
    embeddings
)

faiss.write_index(
    index,
    "faiss_index.index"
)

df.to_csv(
    "papers.csv",
    index=False
)

print("Semua file berhasil disimpan.")

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

In [ ]:
query = "query: large language model transformer bert gpt"

In [ ]:
query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)

query_embedding = np.array(
    query_embedding
).astype('float32')

In [ ]:
top_k = 50

scores, indices = index.search(
    query_embedding,
    top_k
)

In [ ]:
candidates = []

for idx in indices[0]:

    row = df.iloc[idx]

    candidates.append({
        "title": row['title'],
        "abstract": row['abstract'],
        "categories": row['categories'],
        "update_date": row['update_date']
    })

In [ ]:
pairs = [
    (
        query,
        candidate['title']
        + " "
        + candidate['abstract']
    )
    for candidate in candidates
]

In [ ]:
rerank_scores = reranker.predict(
    pairs
)

In [ ]:
for i in range(len(candidates)):

    candidates[i]['rerank_score'] = rerank_scores[i]

In [ ]:
candidates = sorted(
    candidates,
    key=lambda x: x['rerank_score'],
    reverse=True
)

In [ ]:
for paper in candidates[:5]:

    print("=" * 80)
    print()

    print("RERANK SCORE:")
    print(round(paper['rerank_score'], 4))
    print()

    print("TITLE:")
    print(paper['title'])
    print()

    print("CATEGORY:")
    print(paper['categories'])
    print()

    print("UPDATED:")
    print(paper['update_date'])
    print()

    print("ABSTRACT:")
    print(paper['abstract'][:1500])
    print()